# Block folding and outcome averaging

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
from mapd.kinematics import (
    detect_bouts_across_trials, rms_velocity,
    work, positive_effort, holding_cost,
    STATE_REST, STATE_DRIFT, STATE_MOVE,
    _STATE_COLORS, _STATE_LABEL,
)
from mapd.sinq_builders import build_composite_sinq
import h5py
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib.colors as mcolors
import seaborn as sns

import plotly.express as px
from collections import defaultdict
import glob


import pickle
%load_ext autoreload
%autoreload 2
%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

In [ ]:
# sinq2 = Sinq(sinqname='Fig2_learners')
# print(sinq2.__repr__())
# sinq2.df

sinq_0 = Sinq(sinqname='Figure1')
# sinq_1 = Sinq(sinqname='Fig1_control')
# sinq = build_composite_sinq(name='Figure2_learn_v_no_learn_lda',sources=[sinq_0,sinq_1])

# op_df = sinq.display_df(show_tags=True)
# op_df['learn'] = sinq.has_tag('learn')
# op_df['control'] = sinq.has_tag('control')
# op_df[['genotype','tags','learn','control']]

op_df = sinq_0.display_df(show_tags=True)
op_df['learn'] = sinq_0.has_tag('learn')
op_df['control'] = sinq_0.has_tag('control')
# op_df[['genotype','tags','learn','control']]

op_df = op_df.loc[op_df['learn']]
op_df[['genotype','tags','learn','control']]

In [ ]:
sinq2 = Sinq(sinqname='Fig2_learners')

In [ ]:
def fold_blocks(T:Table = None,fig_dir:str = './Figure2',norm:bool = False):
    output_directory = f'{fig_dir}/{T.dfc}/successes_{T.dfc}'
    export_path = f'{fig_dir}/{T.dfc}/exports'

    df = T.df.loc[T.df.index>100,['pyasState','op_cnd_blocks','as_outcome','success','as_duration']]

    df = df.copy()
    df['block_pair'] = (df['op_cnd_blocks'] - 1) // 2 + 1   # pair_id 1..14

    # 2. Align trials relative to the start of each block
    df['trial_in_block'] = df.groupby('op_cnd_blocks').cumcount()
    df['probe_as'] = (df['as_duration']>0) & (df['as_outcome']=='probe')
    df = df.loc[df.trial_in_block<50]
    
    outcome_counts = (
        df.groupby(['pyasState', 'trial_in_block', 'as_outcome'])
        .size()
        .unstack(fill_value=0)
    )
    outcome_counts = outcome_counts.loc[['hi','lo']]
    outcome_counts['max_cnts'] = outcome_counts.sum(axis=1,skipna=True)
    outcome_counts['as_off'] = outcome_counts['as_off'] + outcome_counts['as_off_late']
    outcome_counts['no_as'] = outcome_counts['no_as_no_mv'] + outcome_counts['no_as_mv']
    outcome_counts['to'] = outcome_counts['timeout_fail'] + outcome_counts['timeout']

    success_counts = (
        df.groupby(['pyasState', 'trial_in_block', 'success'])
        .size()
        .unstack(fill_value=0)
    )
    success_counts = success_counts.loc[['hi','lo'],True]
    success_counts = success_counts.to_frame(name='success')
    outcome_counts['success'] = success_counts['success']

    probe_as_counts = (
        df.groupby(['pyasState', 'trial_in_block', 'probe_as'])
        .size()
        .unstack(fill_value=0)
    )
    probe_as_counts = probe_as_counts.loc[['hi','lo']]
    if not True in probe_as_counts.columns:
        probe_as_counts[True] = 0
    probe_as_counts = probe_as_counts.loc[['hi','lo'],True]
    probe_as_counts = probe_as_counts.to_frame(name='probe_as')
    outcome_counts['probe_as'] = probe_as_counts['probe_as']
    
    outcome_counts.to_pickle(path=f'{export_path}/folded_outcome_counts_{T.dfc}.pkl')

    # del fig
    plt.close('all')
    fig = Figure(figsize=(8,8))
    lines = defaultdict(dict)

    colors = {'hi': '#1f77b4', 
            'lo': '#d62728'}
    colors = {'as_off': "#169400", 
              'no_as': "#000cad",
              'no_as_mv': "#6068cb",
              'success': "#009480",
              'to': "#FF0000",
              'probe_as': "#FF30FC",
            }

    hi_x = outcome_counts.xs('hi', level='pyasState').index
    x = {'hi': hi_x,
        'lo': outcome_counts.xs('lo', level='pyasState').index + hi_x[-1] + 1}

    outcome_to_ax = {'as_off':1,
                    'no_as':2,
                    'no_as_mv':2,
                    'success':3,
                    'to':3,
                    'probe_as':3,}

    ax_ids = sorted(set(outcome_to_ax.values()))
    ax_id_to_ax = {}
    for i, ax_id in enumerate(ax_ids, start=1):
        # nrows = len(ax_ids), ncols = 1, index = i
        ax_id_to_ax[ax_id] = fig.add_subplot(len(ax_ids), 1, i)

    for outcome in outcome_to_ax.keys():
        for st in ['hi','lo']:
            ax = ax_id_to_ax[outcome_to_ax[outcome]]
            hi_y = outcome_counts.xs(st, level='pyasState').get(outcome)
            if norm:
                hi_y = hi_y / outcome_counts.xs(st, level='pyasState').get('max_cnts')

            lines[outcome][st], = ax.plot(
                        x[st], hi_y, label=f'{outcome} ({st})', linewidth=1, alpha=0.9, color=colors[outcome]
                    )
            if norm:
                ax.set_ylim([0, 1])
            else:
                ax.set_ylim([0, np.max(outcome_counts.xs(st, level='pyasState').get('max_cnts'))])

    ax_id_to_ax[1].set_ylabel('as_off')
    ax_id_to_ax[2].set_ylabel('no_as')
    ax_id_to_ax[3].set_ylabel('success')
    ax_id_to_ax[3].legend()

    output_directory = f'./Figure2/{T.dfc}'
    os.makedirs(output_directory, exist_ok=True)

    fig.savefig(f'{output_directory}/outcomes_over_blocks_{T.dfc}.svg')
    
    display(fig)

# Fold blocks and average outcomes across blocks

In [ ]:
sinq2.df['blue_toggle_fraction']

In [ ]:
norm = False
total_outcomes = None
total_success = None
total_probe = None
outcome_dict = defaultdict(dict)

outcome_dict = {}
for dfc in reversed(sinq2.df.index):
# for dfc in reversed(sinq2.df.loc[sinq2.df['blue_toggle_fraction']<.8].index):
# for dfc in reversed(sinq2.df.loc[sinq2.df['blue_toggle_fraction']>.8].index):
# for dfc in ['250304_F3_C1','250425_F1_C1']:
    export_path = f'./Figure2/{dfc}/exports'
    pklpat = f'{export_path}/folded_outcome_counts_{dfc}.pkl'
    file_path = glob.glob(pklpat)
    if file_path:
        file_path = file_path[0]
        print(file_path)

    outcome_dict[dfc] = pd.read_pickle(file_path)

    if total_outcomes is None:
        total_outcomes = outcome_dict[dfc].copy()
    else:
        total_outcomes = total_outcomes.add(outcome_dict[dfc], fill_value=0)

norm = True
plt.close('all')
fig = Figure(figsize=(8,8))
lines = defaultdict(dict)

colors = {'hi': '#1f77b4', 
        'lo': '#d62728'}
colors = {'as_off': "#169400", 
            'no_as': "#000cad",
            'no_as_mv': "#6068cb",
            'success': "#009480",
            'to': "#FF0000",
            'probe_as': "#FF30FC",
        }

hi_x = total_outcomes.xs('hi', level='pyasState').index
x = {'hi': hi_x,
    'lo': total_outcomes.xs('lo', level='pyasState').index + hi_x[-1] + 1}

outcome_to_ax = {'as_off':1,
                'no_as':2,
                'no_as_mv':2,
                'success':3,
                'to':3,
                'probe_as':3,}

ax_ids = sorted(set(outcome_to_ax.values()))
ax_id_to_ax = {}
for i, ax_id in enumerate(ax_ids, start=1):
    ax_id_to_ax[ax_id] = fig.add_subplot(len(ax_ids), 1, i)

# for dfc in reversed(sinq2.df.index):
for dfc in outcome_dict.keys():
    for outcome in outcome_to_ax.keys():
        for st in ['hi','lo']:
            outcomes = outcome_dict[dfc]
            ax = ax_id_to_ax[outcome_to_ax[outcome]]
            hi_y = outcomes.xs(st, level='pyasState').get(outcome)
            if norm:
                hi_y = hi_y / outcomes.xs(st, level='pyasState').get('max_cnts')

            ax.scatter(
                        x[st], hi_y, marker='.',s=5, alpha=0.9, c=colors[outcome]
                    )
            if norm:
                ax.set_ylim([0, 1])
            else:
                ax.set_ylim([0, np.max(outcomes.xs(st, level='pyasState').get('max_cnts'))])


for outcome in outcome_to_ax.keys():
    for st in ['hi','lo']:
        ax = ax_id_to_ax[outcome_to_ax[outcome]]
        hi_y = total_outcomes.xs(st, level='pyasState').get(outcome)
        if norm:
            hi_y = hi_y / total_outcomes.xs(st, level='pyasState').get('max_cnts')

        lines[outcome][st], = ax.plot(
                    x[st], hi_y, label=f'{outcome} ({st})', linewidth=1, alpha=0.9, color=colors[outcome]
                )
        if norm:
            ax.set_ylim([0, 1])
        else:
            ax.set_ylim([0, np.max(total_outcomes.xs(st, level='pyasState').get('max_cnts'))])

ax_id_to_ax[1].set_ylabel('as_off')
ax_id_to_ax[2].set_ylabel('no_as')
ax_id_to_ax[3].set_ylabel('success')
ax_id_to_ax[3].legend()
fig.savefig(f'./Figure2/outcomes_over_blocks_all.svg',format='svg')
# fig.savefig(f'./Figure2/outcomes_over_blocks_red.svg',format='svg')
# fig.savefig(f'./Figure2/outcomes_over_blocks_blue.svg',format='svg')
display(fig)

## Get the mean of each category of the last 30 trials for each fly

In [ ]:
import random

outcome_dict = {}
plt.close('all')
fig = Figure(figsize=(8,8))
fly_means = defaultdict(dict)
# fly_means = defaultdict(lambda: defaultdict(dict))

outcome_to_ax = {'as_off':1,
                'no_as':2,
                # 'no_as_mv':2,
                # 'success':3,
                'to':3,
                # 'probe_as':3,
                }

for dfc in reversed(sinq2.df.index):
# for dfc in ['250304_F3_C1']:
    export_path = f'./Figure2/{dfc}/exports'
    pklpat = f'{export_path}/folded_outcome_counts_{dfc}.pkl'
    file_path = glob.glob(pklpat)
    if file_path:
        file_path = file_path[0]
        print(file_path)

    outcome_dict[dfc] = pd.read_pickle(file_path)
    for outcome in outcome_to_ax.keys():
        fly_means[outcome]['hi'] = []
        fly_means[outcome]['lo'] = []


ax_ids = sorted(set(outcome_to_ax.values()))
ax_id_to_ax = {}
for i, ax_id in enumerate(ax_ids, start=1):
    ax_id_to_ax[ax_id] = fig.add_subplot(len(ax_ids), 1, i)
    ax_id_to_ax[ax_id].set_ylim([0, 1])

# for dfc in reversed(sinq2.df.index):
for dfc in outcome_dict.keys():
    outcomes = outcome_dict[dfc]
    for st in ['hi','lo']:
        last30 = outcomes.loc[(st,[i for i in range(20,44)]),:]
        for outcome in outcome_to_ax.keys():
            ax = ax_id_to_ax[outcome_to_ax[outcome]]
            hi_y = last30.get(outcome) / last30.xs(st, level='pyasState').get('max_cnts')
            hi_y = hi_y.mean()
            x = 1 if st=='hi' else 2
            x = x +random.uniform(-.1, .1)
            ax.scatter(x, hi_y, marker='o',s=10, alpha=0.9,label=dfc)

            fly_means[outcome][st].append(hi_y)

for outcome in fly_means.keys():
    out_means = fly_means[outcome]
    for st in out_means.keys():
        ax = ax_id_to_ax[outcome_to_ax[outcome]]
        x = 1 if st=='hi' else 2
        x = x + .2
        st_means = out_means[st]
        print(st_means)
        ax.boxplot(st_means, vert=True, positions=[x])
        # ax.boxplot(st_means)

display(fig)

In [ ]:
fly_means

In [ ]:
outcomes.loc[('hi',[i for i in range(20,44)]),:].mean()